# Surface worlds

`surface_world` builds a panel world from a `SurfaceSpec`: doses from a `DosePlan`, hierarchical
intercepts, nuisance, noise — through the *same* `build → prepare → value` path the model uses
(one `forward()`). It carries its own truth: parameters, the noise-free mean, marginal effects.
`arms_world` is the single-period special case.

In [ ]:
import numpy as np

from axiom.core import D, Outcome, Treatment, is_failure
from axiom.surface import FourierSeasonality, GeometricCarryover, HillKernel, NuisanceSet, Surface, check_linearization
from axiom.sim import (
    DoseDistribution, DosePlan, Horizon, SurfaceWorld, TruthMode, arms_world, draw_doses, long_frame,
    panel_from_arrays, surface_world, true_parameters, unit_labels,
)

In [ ]:
dist: DoseDistribution = "lognormal"
plan = DosePlan(distribution=dist, scale=50.0, spread=0.6, zero_fraction=0.1)
print(draw_doses(plan, 2, 6, seed=0).round(1))
print(unit_labels(3))

In [ ]:
mode: TruthMode = "centre"
world: SurfaceWorld = surface_world(
    n_units=4, n_periods=26, treatments=("a", "b"),
    kernels={"a": HillKernel(reference_dose=50.0), "b": HillKernel(reference_dose=20.0)},
    carryover={"a": GeometricCarryover(max_lag=4)},
    doses={"a": plan, "b": DosePlan(scale=20.0, spread=0.4)},
    nuisance=NuisanceSet(terms=(FourierSeasonality(period=13.0, order=1),)),
    intercept="hierarchical", truth_mode=mode, noise_sd=0.5, seed=3,
)
print(world.panel)
print({k: np.round(v, 3) for k, v in world.theta.items() if k in world.structural})
print("noise sd:", world.noise.std().round(3), "| provenance:", {k: world.provenance[k] for k in ("seed", "truth_mode")})

The world's mean is exactly what the `Surface` computes from its data and truth, and the
linearization invariant holds on world data.

In [ ]:
surface = Surface(world.spec)
print(np.allclose(surface.forward(world.data, world.theta), world.mean), check_linearization(surface, world.data, world.theta))
print("total response at observed doses, unit 0, first 5 periods:", world.total_response()[0, :5].round(3))
horizon: Horizon = "total"
m = world.marginal("b", horizon=horizon)
print(m if is_failure(m) else m[0, :5].round(4))

In [ ]:
theta_prior = true_parameters(world.model, structural=world.structural, mode="prior", seed=11)
print({k: np.round(v, 3) for k, v in theta_prior.items() if k in ("k_a", "s_a", "lam_a")})

## Arms and hand-built panels

In [ ]:
from axiom.display import show

arms = arms_world(n_units=12, treatments=("dose",), doses={"dose": np.linspace(0, 100, 12)}, seed=0)
print(arms.panel, arms.n_periods)
frame = long_frame({"x": [[1.0, 2.0], [3.0, 4.0]], "y": [[0.5, 0.6], [0.7, 0.8]]}, units=("u0", "u1"))
print(frame)
panel = panel_from_arrays(
    doses={"x": [[1.0, 2.0], [3.0, 4.0]]}, outcome=[[0.5, 0.6], [0.7, 0.8]],
    treatments=[Treatment(name="x", dimension=D.currency, unit="USD")],
    outcome_entity=Outcome(name="y", dimension=D.outcome), units=("u0", "u1"),
)
show(panel)

## Seeing it

Everything above prints. `axiom.display` renders the same results as cards —
coloured when `rich` is installed, aligned plain text when it is not — and
`enable()` makes that the default for the rest of the notebook, so a bare result
on the last line of a cell renders itself.

`axiom.viz` draws the figure this subpackage's results are actually about.

In [ ]:
import numpy as np

from axiom.display import enable, show
from axiom.sim import arms_world
from axiom.viz import recovery

enable()

arms = arms_world(n_units=12, treatments=("dose",), doses={"dose": np.linspace(0, 100, 12)}, seed=0)
show(arms.panel)

truth = {"beta": 2.0, "gamma": -0.75, "sigma": 1.0}
estimated = {"beta": 1.94, "gamma": -0.81, "sigma": 1.06}
recovery(truth, estimated)

The picture every recovery test is implicitly making: distance from the diagonal is bias.